# ECC/EMC — Kaggle Training Notebook

Re-implementation of "To Err Like Human" (NeurIPS 2024) — ECC/EMC metrics, ListMLE loss, EMC-adaptive FixMatch.

## Before you run
1. **Enable GPU:** Settings → Session options → GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Add your private FI dataset and the public EmoSet-118K dataset in the notebook sidebar.
   3. **Replace the dataset slugs below** (the `YOUR_USERNAME` placeholders) with your actual Kaggle usernames/dataset slugs.

## What this notebook does
- Installs `ecc-ver` from the uploaded code
- Symlinks Kaggle input datasets into the expected `data/` layout
- Runs supervised ResNet50 on FI (alpha = 0 / 0.2 / 1.0, Table 4)
- Runs semi-supervised FixMatch with EMC-adaptive threshold (Section 4.2.1)
- Saves checkpoints every epoch, so you can resume across 12-hour session limits
- Writes results to `/kaggle/working/runs/` (download before the session ends)

In [ ]:
# ===================== CONFIGURE THESE =====================
# Your private FI dataset (resized 256x256, made with scripts/resize_and_split.py)
FI_DATASET_SLUG = "your-username/fi-dataset-256"   # <-- REPLACE

# Public EmoSet-118K dataset (you said you found it on Kaggle)
EMOSET_DATASET_SLUG = "username/emoset-118k"          # <-- REPLACE

# Which experiments to run
RUN_SUPERVISED = True
RUN_SSL = True
# ==========================================================

print("FI dataset:", FI_DATASET_SLUG)
print("EmoSet dataset:", EMOSET_DATASET_SLUG)

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone / install ecc-ver
# Option A: upload your code as a Kaggle dataset and uncomment:
# !pip -q install /kaggle/input/your-code-dataset/ecc-ver

# Option B: clone from GitHub (uncomment if you push to GitHub):
# !pip -q install git+https://github.com/YOUR_USERNAME/ecc-ver

# Option C (default): copy the repo into the notebook working dir
import os, shutil
repo_src = "/kaggle/input/your-code-dataset"  # <-- if you uploaded code as a dataset
if os.path.isdir(repo_src):
    shutil.copytree(repo_src + "/ecc-ver", "/kaggle/working/ecc-ver", dirs_exist_ok=True)
    os.chdir("/kaggle/working/ecc-ver")
else:
    print("No code dataset found at", repo_src)
    print("Upload ecc-ver as a Kaggle dataset, or clone it from GitHub.")

In [ ]:
# Symlink datasets into the expected data/ layout
import os

os.makedirs("/kaggle/working/ecc-ver/data", exist_ok=True)

# FI
fi_input = f"/kaggle/input/{FI_DATASET_SLUG.split('/')[1]}" if "/" in FI_DATASET_SLUG else f"/kaggle/input/{FI_DATASET_SLUG}"
print("Looking for FI at:", fi_input)
# Find the FI/ and FI_splits/ subdirectories inside the mounted dataset
for root, dirs, files in os.walk(fi_input):
    for d in dirs:
        if d in ("FI", "FI_splits"):
            src = os.path.join(root, d)
            dst = f"/kaggle/working/ecc-ver/data/{d}"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked {d} -> {dst}")

# EmoSet
emoset_input = f"/kaggle/input/{EMOSET_DATASET_SLUG.split('/')[1]}" if "/" in EMOSET_DATASET_SLUG else f"/kaggle/input/{EMOSET_DATASET_SLUG}"
print("Looking for EmoSet at:", emoset_input)
for root, dirs, files in os.walk(emoset_input):
    for d in dirs:
        if d in ("EmoSet", "EmoSet_splits", "image"):
            src = os.path.join(root, d)
            dst = f"/kaggle/working/ecc-ver/data/{d}"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked {d} -> {dst}")

print("\nData layout:")
!find /kaggle/working/ecc-ver/data -maxdepth 3 | head -30

In [ ]:
# Install ecc-ver package
!cd /kaggle/working/ecc-ver && pip -q install -e .

In [ ]:
# Quick smoke test (CPU, no data needed)
!cd /kaggle/working/ecc-ver && python -m pytest -q

In [ ]:
# ===================== SUPERVISED (FI, ResNet50, Table 4) =====================
if RUN_SUPERVISED:
    import subprocess
    
    base = ["python", "-m", "ecc.supervised", "--config", "configs/fi_resnet50.yaml"]
    
    # alpha = 0 (pure CE)
    print("=== alpha=0 ===")
    !cd /kaggle/working/ecc-ver && python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=0.0 output_dir=/kaggle/working/runs/fi_r50_a0
    
    # alpha = 0.2
    print("=== alpha=0.2 ===")
    !cd /kaggle/working/ecc-ver && python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=0.2 output_dir=/kaggle/working/runs/fi_r50_a0.2
    
    # alpha = 1.0
    print("=== alpha=1.0 ===")
    !cd /kaggle/working/ecc-ver && python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=1.0 output_dir=/kaggle/working/runs/fi_r50_a1.0

print("Supervised done.")

In [ ]:
# ===================== SEMI-SUPERVISED (FI, Section 4.2.1) =====================
if RUN_SSL:
    import subprocess
    
    # 800 labeled, EMC-adaptive threshold
    print("=== SSL n=800, rule=ratio (EMC-adaptive) ===")
    !cd /kaggle/working/ecc-ver && python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=800 ssl.rule=ratio output_dir=/kaggle/working/runs/ssl_fi_n800_ratio
    
    # 800 labeled, fixed threshold (FixMatch baseline)
    print("=== SSL n=800, rule=fixed (FixMatch baseline) ===")
    !cd /kaggle/working/ecc-ver && python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=800 ssl.rule=fixed output_dir=/kaggle/working/runs/ssl_fi_n800_fixed

print("SSL done.")

In [ ]:
# List all output files for download
!find /kaggle/working/runs -type f | sort

## Resuming after a session timeout

Kaggle sessions are capped at ~12 hours. If training is interrupted, simply re-run the same cells —
each training script auto-detects `checkpoint.pt` in its `output_dir` and resumes from the last saved epoch.

To force a specific checkpoint, add `--set resume_from=/kaggle/working/runs/<run>/checkpoint.pt`.

## Download results

Before the session ends, download `/kaggle/working/runs/` (contains `best.pt`, `results.json`, `history.jsonl`, `config.json`).